In [0]:
from decimal import Decimal

from pyspark.sql import DataFrame, Window, functions as F

GOLD_TABLES = [
    "daily_sales",
    "monthly_category_sales",
    "geography_sales",
    "product_performance",
    "customer_summary",
    "payment_summary",
]

# tables whose net_revenue column must add up to the silver total (used for reconciliation)
RECONCILE_TABLES = [
    "daily_sales", "monthly_category_sales", "geography_sales",
    "product_performance", "payment_summary",
]

HIGH_SPEND = 10000
MEDIUM_SPEND = 3000


def _money(col):
    return F.round(col, 2).cast("decimal(18,2)")


def revenue_orders(df: DataFrame) -> DataFrame:
    return df.where(F.col("is_revenue_order"))


def total_net_revenue(df: DataFrame):
    value = revenue_orders(df).agg(F.sum("net_amount")).collect()[0][0]
    return value if value is not None else Decimal("0.00")


def daily_sales(df: DataFrame) -> DataFrame:
    return (
        revenue_orders(df).groupBy("order_date").agg(
            F.countDistinct("order_id").alias("orders"),
            F.sum("quantity").alias("units_sold"),
            _money(F.sum("gross_amount")).alias("gross_amount"),
            _money(F.sum("discount_amount")).alias("discount_amount"),
            _money(F.sum("net_amount")).alias("net_revenue"),
        )
        .withColumn("avg_order_value", _money(F.col("net_revenue") / F.col("orders")))
        .withColumn("order_year", F.year("order_date"))
        .withColumn("order_month", F.month("order_date"))
        .orderBy("order_date")
    )


def monthly_category_sales(df: DataFrame) -> DataFrame:
    agg = revenue_orders(df).groupBy("order_year", "order_month", "category").agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("quantity").alias("units_sold"),
        _money(F.sum("gross_amount")).alias("gross_amount"),
        _money(F.sum("discount_amount")).alias("discount_amount"),
        _money(F.sum("net_amount")).alias("net_revenue"),
    )
    w = Window.partitionBy("category").orderBy("order_year", "order_month")
    prev = F.lag("net_revenue").over(w)
    return (
        agg.withColumn("prev_month_net_revenue", prev)
           .withColumn("mom_growth_pct",
                       F.when(F.col("prev_month_net_revenue") > 0,
                              F.round((F.col("net_revenue") - F.col("prev_month_net_revenue"))
                                      / F.col("prev_month_net_revenue") * 100, 2)))
           .orderBy("order_year", "order_month", "category")
    )


def geography_sales(df: DataFrame) -> DataFrame:
    agg = (
        revenue_orders(df)
        .withColumn("country", F.coalesce("country", F.lit("Unknown")))
        .withColumn("city", F.coalesce("city", F.lit("Unknown")))
        .groupBy("country", "city").agg(
            F.countDistinct("order_id").alias("orders"),
            F.sum("quantity").alias("units_sold"),
            _money(F.sum("net_amount")).alias("net_revenue"),
        )
    )
    total = F.sum("net_revenue").over(Window.partitionBy())
    return (
        agg.withColumn("revenue_share_pct", F.when(total > 0, F.round(F.col("net_revenue") / total * 100, 2)))
           .withColumn("revenue_rank", F.rank().over(Window.orderBy(F.col("net_revenue").desc())))
           .orderBy("revenue_rank", "country", "city")
    )


def product_performance(df: DataFrame) -> DataFrame:
    agg = revenue_orders(df).groupBy("product_name", "category").agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("quantity").alias("units_sold"),
        _money(F.sum("net_amount")).alias("net_revenue"),
        _money(F.avg("unit_price")).alias("avg_unit_price"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
    )
    return (
        agg.withColumn("revenue_rank", F.rank().over(Window.orderBy(F.col("net_revenue").desc())))
           .withColumn("rank_in_category",
                       F.rank().over(Window.partitionBy("category").orderBy(F.col("net_revenue").desc())))
           .orderBy("revenue_rank", "product_name")
    )


def customer_summary(df: DataFrame) -> DataFrame:
    """One row per customer. Key = email, else phone, else name (silver has no customer id)."""
    key = F.coalesce(F.col("customer_email"), F.col("phone_number"), F.lower(F.col("customer_name")))
    agg = (
        revenue_orders(df).withColumn("customer_key", key).where(F.col("customer_key").isNotNull())
        .groupBy("customer_key").agg(
            F.max("customer_name").alias("customer_name"),
            F.max("country").alias("country"),
            F.countDistinct("order_id").alias("orders"),
            _money(F.sum("net_amount")).alias("net_spend"),
            F.min("order_date").alias("first_order_date"),
            F.max("order_date").alias("last_order_date"),
        )
    )
    return (
        agg.withColumn("avg_order_value", _money(F.col("net_spend") / F.col("orders")))
           .withColumn("customer_segment",
                       F.when(F.col("net_spend") >= HIGH_SPEND, "High")
                        .when(F.col("net_spend") >= MEDIUM_SPEND, "Medium")
                        .otherwise("Low"))
           .orderBy(F.col("net_spend").desc(), "customer_key")
    )


def payment_summary(df: DataFrame) -> DataFrame:
    agg = df.groupBy("payment_method").agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum(F.when(F.col("is_revenue_order"), 1).otherwise(0)).alias("revenue_orders"),
        F.sum(F.when(~F.col("is_revenue_order"), 1).otherwise(0)).alias("cancelled_or_returned_orders"),
        _money(F.sum(F.when(F.col("is_revenue_order"), F.col("net_amount")).otherwise(F.lit(0)))).alias("net_revenue"),
    )
    total = F.sum("net_revenue").over(Window.partitionBy())
    return (
        agg.withColumn("cancel_return_rate_pct",
                       F.round(F.col("cancelled_or_returned_orders") / F.col("orders") * 100, 2))
           .withColumn("revenue_share_pct", F.when(total > 0, F.round(F.col("net_revenue") / total * 100, 2)))
           .orderBy(F.col("net_revenue").desc(), "payment_method")
    )


def build_gold(silver_df: DataFrame) -> dict:
    """Return {gold_table_name: DataFrame}."""
    return {
        "daily_sales": daily_sales(silver_df),
        "monthly_category_sales": monthly_category_sales(silver_df),
        "geography_sales": geography_sales(silver_df),
        "product_performance": product_performance(silver_df),
        "customer_summary": customer_summary(silver_df),
        "payment_summary": payment_summary(silver_df),
    }